In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

rows = 4
cols = 4
start_state = (0, 0)
goal_state = (3, 3)
blocked_states = {(1, 1), (2, 1)}

actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
action_names = ["Up", "Down", "Left", "Right"]

q_table = np.zeros((rows * cols, len(actions)))

def state_index(position):
    return position[0] * cols + position[1]

def environment_step(position, action):
    row_change, col_change = actions[action]

    next_position = (
        min(max(position[0] + row_change, 0), rows - 1),
        min(max(position[1] + col_change, 0), cols - 1)
    )

    if next_position in blocked_states:
        next_position = position

    terminated = next_position == goal_state
    reward = 10 if terminated else -1

    return next_position, reward, terminated

print("Grid World Size:", rows, "x", cols)
print("Start State:", start_state)
print("Goal State:", goal_state)
print("Blocked States:", blocked_states)

In [ ]:
learning_rate = 0.1
discount_factor = 0.9
epsilon = 1.0
epsilon_decay = 0.995
minimum_epsilon = 0.05

episodes = 2000
maximum_steps = 100
episode_rewards = []

for episode in range(episodes):
    current_state = start_state
    total_reward = 0

    for step in range(maximum_steps):
        current_index = state_index(current_state)

        if np.random.random() < epsilon:
            action = np.random.randint(len(actions))
        else:
            action = int(np.argmax(q_table[current_index]))

        next_state, reward, terminated = environment_step(
            current_state,
            action
        )

        next_index = state_index(next_state)

        if terminated:
            target_value = reward
        else:
            target_value = (
                reward
                + discount_factor * np.max(q_table[next_index])
            )

        q_table[current_index, action] += learning_rate * (
            target_value - q_table[current_index, action]
        )

        current_state = next_state
        total_reward += reward

        if terminated:
            break

    epsilon = max(minimum_epsilon, epsilon * epsilon_decay)
    episode_rewards.append(total_reward)

print("Training Episodes:", episodes)
print("Final Epsilon:", round(epsilon, 4))
print(
    "Average Reward in Last 100 Episodes:",
    round(np.mean(episode_rewards[-100:]), 2)
)

In [ ]:
current_state = start_state
learned_path = [current_state]

for step in range(maximum_steps):
    current_index = state_index(current_state)
    action = int(np.argmax(q_table[current_index]))

    next_state, reward, terminated = environment_step(
        current_state,
        action
    )

    if next_state == current_state:
        available_actions = [
            index for index, (dr, dc) in enumerate(actions)
            if (
                0 <= current_state[0] + dr < rows
                and 0 <= current_state[1] + dc < cols
                and (
                    current_state[0] + dr,
                    current_state[1] + dc
                ) not in blocked_states
            )
        ]

        if available_actions:
            action = max(
                available_actions,
                key=lambda index: q_table[current_index, index]
            )
            next_state, reward, terminated = environment_step(
                current_state,
                action
            )

    current_state = next_state
    learned_path.append(current_state)

    if terminated:
        break

print("Learned Path:", learned_path)
print("Goal Reached:", current_state == goal_state)

plt.figure(figsize=(8, 4))
plt.plot(episode_rewards)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Q-Learning Training Performance")
plt.grid(alpha=0.25)
plt.show()

grid = np.zeros((rows, cols))

for row, col in blocked_states:
    grid[row, col] = -1

grid[goal_state] = 2

plt.figure(figsize=(5, 5))
plt.imshow(grid, cmap="coolwarm")

for row in range(rows):
    for col in range(cols):
        position = (row, col)

        if position == start_state:
            label = "S"
        elif position == goal_state:
            label = "G"
        elif position in blocked_states:
            label = "X"
        elif position in learned_path:
            label = "●"
        else:
            label = ""

        plt.text(
            col,
            row,
            label,
            ha="center",
            va="center",
            fontsize=14
        )

plt.xticks(range(cols))
plt.yticks(range(rows))
plt.title("Optimal Policy in Grid World")
plt.grid(alpha=0.25)
plt.show()